# Embryo workflow

Classify single or batch measurement CSVs with RF / SVM, or analyze fixed-embryo TIFFs with ROI ZIPs using Fiji neighbour counting and Python distribution plots.

Select `EmbryoAnalyser/.venv/Scripts/python.exe` as the kernel. Start from the project root or this notebook's directory. Edit the input configuration below; results default to `outputs/notebook/`.

In [ ]:
from pathlib import Path
import json
import sys

working = Path.cwd().resolve()
ROOT = next((path for path in [working, *working.parents]
             if (path / 'EmbryoAnalyser/workflow.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start the notebook from the project root or EmbryoAnalyser directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Image
from EmbryoAnalyser.workflow import train_workflow, classify_workflow, distribution_workflow
from EmbryoAnalyser.neighbours import run_neighbours

print('Project:', ROOT)


## Input configuration

Each input list accepts one file, multiple files, or a recursive directory. `MODELS` can be `both`, `rf`, or `svm`. The default training input explicitly selects the gap43 `train` directory.

Set `RETRAIN=False` to reuse a saved model. The two `RUN_...` switches independently select classification and image analysis. This notebook requires existing ROI ZIPs; use `staged_workflow.ipynb` for automatic segmentation.

In [ ]:
TRAIN_INPUTS = [ROOT / 'dataset/raw_dataset/gap43-mCherry/train']
CSV_INPUTS = [ROOT / 'dataset/raw_dataset/E-CadGFP']
IMAGE_INPUTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/Composite.tif']

MODELS = 'both'
OUTPUT = ROOT / 'outputs/notebook'
MODEL_FILE = OUTPUT / 'training/models.joblib'
RETRAIN = True
RUN_CLASSIFICATION = True
RUN_FIJI = True
FIJI_TIMEOUT = None  # No fixed limit; optional whole-batch time limit in seconds.


## CSV classification

Each CSV represents one embryo. Cleaning, binning, and classifier settings follow the original notebook. Labels are `0=control` and `1=mutant`. Prediction does not require known labels; only recognized true labels contribute to report metrics.

In [ ]:
if RUN_CLASSIFICATION:
    if RETRAIN:
        trained = train_workflow(TRAIN_INPUTS, OUTPUT / 'training', model=MODELS)
        MODEL_FILE = trained['model_file']
        print('Training embryos:', trained['report']['training_embryos'])
        print('Model saved to:', MODEL_FILE)
    classified = classify_workflow(CSV_INPUTS, MODEL_FILE, OUTPUT / 'classification')
    display(classified['predictions'])
    print('Prediction CSV:', classified['predictions_csv'])
    print('Evaluation metrics:', json.dumps(classified['report']['metrics'], ensure_ascii=False, indent=2))


## Fiji neighbour counting and Python distributions

Images and ROIs are copied to a new run directory. Counting runs in an isolated Fiji UI and exits on completion. The macro working copy replaces only its directory prompt. Outputs include per-cell counts, frequency and percentage CSVs, PNG/PDF plots, and a batch summary.

In [ ]:
if RUN_FIJI:
    neighbours = run_neighbours(IMAGE_INPUTS, OUTPUT / 'neighbours',
                                timeout=FIJI_TIMEOUT, show=False)
    display(pd.read_csv(neighbours['summary_csv']))
    for sample in neighbours['samples']:
        print('Input:', sample['input_image'])
        display(pd.read_csv(sample['distribution_csv']))
        display(Image(filename=sample['png']))
    print('Batch summary:', neighbours['summary_csv'])


## Optional: plot existing counting CSVs

Existing `slow_neighbour_counting.csv` files can be plotted directly in Python without rerunning Fiji. Set `EXISTING_COUNTS` to a file, a list of files, or a directory, then enable `PLOT_EXISTING_COUNTS=True`.

In [ ]:
PLOT_EXISTING_COUNTS = False
EXISTING_COUNTS = [ROOT / 'dataset/fixed_EM/processed/del15/s9_1/slow_neighbour_counting.csv']

if PLOT_EXISTING_COUNTS:
    plotted = distribution_workflow(EXISTING_COUNTS, OUTPUT / 'existing_distributions')
    for sample in plotted['samples']:
        print('Input:', sample['input_csv'])
        display(pd.read_csv(sample['distribution_csv']))
        display(Image(filename=sample['png']))
    print('Summary:', plotted['summary_csv'])
